# 💬 LangChain Message Types — HumanMessage, SystemMessage, AIMessage, ToolMessage
### Understanding all 4 message types with your Capgemini LLM Setup
> **Python 3.11 | LangChain | ChatOpenAI | .env config**


## Cell 1 — Install & Imports

In [ ]:
%pip install langchain-openai python-dotenv -q

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
import os
from dotenv import load_dotenv

load_dotenv()
print("✅ Ready!")

## Cell 2 — Base LLM Setup

In [ ]:
llm = ChatOpenAI(
    model=os.getenv("MODEL"),
    base_url=os.getenv("API_URL"),
    api_key=os.getenv("API_KEY"),
    temperature=0.7,
)

print("✅ LLM configured!")

## 💬 What are these Message Types?

| Message Type  | Role      | Purpose                                          |
|---------------|-----------|--------------------------------------------------|
| SystemMessage | system    | Sets the behaviour/persona of the LLM            |
| HumanMessage  | user      | The user's input / question                      |
| AIMessage     | assistant | The LLM's response                               |
| ToolMessage   | tool      | Result returned by a tool back to the LLM        |


## Cell 4 — HumanMessage

In [ ]:
# HumanMessage = what the user says to the LLM

human_msg = HumanMessage(content="What is the capital of France?")

print("📦 Message Object:")
print(f"   Type    : {type(human_msg).__name__}")
print(f"   Role    : {human_msg.type}")
print(f"   Content : {human_msg.content}")

print("\n🤖 LLM Response:")
response = llm.invoke([human_msg])
print(f"   {response.content}")

## Cell 5 — SystemMessage

In [ ]:
# SystemMessage = instructions that shape the LLM's persona/behaviour
# Always placed FIRST in the message list

system_msg = SystemMessage(content=(
    "You are a pirate assistant. "
    "You must answer every question like a pirate. "
    "Use pirate slang and end every response with 'Arrr!'"
))

human_msg = HumanMessage(content="What is the capital of France?")

print("📦 System Message:")
print(f"   Type    : {type(system_msg).__name__}")
print(f"   Role    : {system_msg.type}")
print(f"   Content : {system_msg.content}")

print("\n🤖 LLM Response (with persona):")
response = llm.invoke([system_msg, human_msg])
print(f"   {response.content}")

## Cell 6 — AIMessage

In [ ]:
# AIMessage = the LLM's own response
# You can also manually create AIMessages to simulate conversation history

ai_msg = AIMessage(content="The capital of France is Paris.")

print("📦 AI Message Object:")
print(f"   Type    : {type(ai_msg).__name__}")
print(f"   Role    : {ai_msg.type}")
print(f"   Content : {ai_msg.content}")

# AIMessage is what llm.invoke() always returns
print("\n🔁 llm.invoke() also returns an AIMessage:")
response = llm.invoke([HumanMessage(content="Hello!")])
print(f"   Returned type : {type(response).__name__}")
print(f"   Content       : {response.content}")

## Cell 7 — Multi-Turn Conversation (System + Human + AI)

In [ ]:
# This is how a real chat history looks —
# SystemMessage sets the context, then Human/AI alternate

conversation = [
    SystemMessage(content="You are a helpful data science tutor. Keep answers short."),
    HumanMessage(content="What is overfitting?"),
    AIMessage(content="Overfitting is when a model learns the training data too well, including noise, and performs poorly on new data."),
    HumanMessage(content="How do I fix it?"),
]

print("📋 Conversation History:")
for msg in conversation:
    role = type(msg).__name__
    print(f"   [{role}]: {msg.content}")

print("\n🤖 LLM Response (continuing the conversation):")
response = llm.invoke(conversation)
print(f"   [AIMessage]: {response.content}")

## Cell 8 — ToolMessage Setup (Define Tool)

In [ ]:
# ToolMessage = the result returned by a tool back to the LLM
# It must always reference a tool_call_id from an AIMessage that requested the tool

@tool
def get_weather(city: str) -> str:
    """Returns the current weather for a given city."""
    dummy_weather = {
        "Mumbai"  : "32°C, Humid, Partly Cloudy",
        "London"  : "14°C, Rainy",
        "New York": "22°C, Sunny",
    }
    return dummy_weather.get(city, f"Weather data not available for {city}.")

tools = [get_weather]
llm_with_tools = llm.bind_tools(tools)

print("✅ Tool registered :", get_weather.name)
print("✅ LLM bound with tools!")

## Cell 9 — ToolMessage Full Flow

In [ ]:
# Step 1: User asks something that needs a tool
human_msg = HumanMessage(content="What is the weather in Mumbai?")

print("=" * 55)
print("Step 1 — HumanMessage sent to LLM")
print("=" * 55)
print(f"   {human_msg.content}\n")

# Step 2: LLM decides to call the tool → returns AIMessage with tool_calls
ai_response = llm_with_tools.invoke([human_msg])

print("Step 2 — AIMessage received (with tool call request)")
print("=" * 55)
print(f"   Type       : {type(ai_response).__name__}")
print(f"   Tool Calls : {ai_response.tool_calls}\n")

# Step 3: Execute the tool and wrap result in ToolMessage
tool_call    = ai_response.tool_calls[0]
tool_result  = get_weather.invoke(tool_call["args"])
tool_call_id = tool_call["id"]

tool_msg = ToolMessage(
    content=tool_result,
    tool_call_id=tool_call_id,
)

print("Step 3 — ToolMessage created from tool result")
print("=" * 55)
print(f"   Type        : {type(tool_msg).__name__}")
print(f"   Tool Call ID: {tool_msg.tool_call_id}")
print(f"   Content     : {tool_msg.content}\n")

# Step 4: Send full conversation back to LLM for final answer
final_response = llm_with_tools.invoke([
    human_msg,
    ai_response,
    tool_msg,
])

print("Step 4 — Final AIMessage after tool result")
print("=" * 55)
print(f"   🤖 {final_response.content}")

## Cell 10 — Full Message Chain Summary

In [ ]:
all_messages = [
    human_msg,
    ai_response,
    tool_msg,
    final_response,
]

print("📋 Complete Message Chain\n")
print(f"{'#':<4} {'Message Type':<16} {'Content (truncated)'}")
print("-" * 65)

for i, msg in enumerate(all_messages):
    content = str(msg.content)[:50].replace("\n", " ")
    print(f"{i+1:<4} {type(msg).__name__:<16} {content}")

## ✅ Summary

| Message Type  | When to use                                       | Created by  |
|---------------|---------------------------------------------------|-------------|
| SystemMessage | Set LLM persona/behaviour at the start            | You         |
| HumanMessage  | Every user input/question                         | You         |
| AIMessage     | LLM response (also used to build chat history)    | LLM / You   |
| ToolMessage   | Return tool result back to LLM after a tool call  | You         |

### 🔑 Key Flow
```
SystemMessage → HumanMessage → AIMessage → ToolMessage → AIMessage
   (persona)       (user)        (thinks)    (tool runs)  (final answer)
```
